In [1]:
import os

os.chdir("../..")

In [2]:
from datetime import datetime
import pynmea2
from src.kelder_api.components.gps_new.types import GPSStatus
from src.kelder_api.components.gps_new.models import (
    GPGSVSatellitesInView,
    GPRMCRecommendedCourse,
    GPGSAActiveSatellites,
)

nmea_sentences_active = {
    "GPVTG": "$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64",
    "GPGSA": "$GPGSA,A,3,30,20,13,14,11,,,,,,,,4.17,3.14,2.74*01",
    "GPGSV_1": "$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70",
    "GPGSV_2": "$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E",
    "GPGSV_3": "$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B",
    "GPGSV_4": "$GPGSV,4,4,14,27,04,019,,30,69,085,*74",
    "GPGLL": "$GPGLL,,,,,202233.00,V,N*48",
    "GPRMC": "$GPRMC,200517.00,A,5211.77084,N,00213.26972,W,0.117,,030625,,,A*65",
}

nmea_sentences_void = {
    "GPVTG": "$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64",
    "GPGSA": "$GPGSA,A,1,,,,,,,,,,,,,99.99,99.99,99.99*30 ",
    "GPGSV_1": "$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70",
    "GPGSV_2": "$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E",
    "GPGSV_3": "$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B",
    "GPGSV_4": "$GPGSV,4,4,14,27,04,019,,30,69,085,*74",
    "GPGLL": "$GPGLL,,,,,202233.00,V,N*48",
    "GPRMC": "$GPRMC,202234.00,V,,,,,,,090625,,,N*70",
}

In [3]:
nmea_parsed = pynmea2.parse(nmea_sentences_active["GPRMC"])

nmea_datetime = datetime.combine(nmea_parsed.datestamp, nmea_parsed.timestamp)

nmea_parsed.status

# GPSStatus.ACTIVE
GPRMCRecommendedCourse.from_nmea(nmea_parsed)

GPRMCRecommendedCourse(timestamp=datetime.datetime(2025, 6, 3, 20, 5, 17, tzinfo=datetime.timezone.utc), status=<GPSStatus.ACTIVE: 'A'>, latitude_nmea='5211.77084', longitude_nmea='00213.26972')

In [4]:
nmea_parsed = pynmea2.parse(nmea_sentences_active["GPGSA"])
print(nmea_parsed.hdop)
x = getattr(nmea_parsed, "sv_id01")
print(x)
# for field in nmea_parsed.fields:
#     if field[1].startswith('sv_id'):
#         pass#print(nmea_parsed.__dict__["data"][field])

prns = [
    getattr(nmea_parsed, sat_prn[1])
    for sat_prn in nmea_parsed.fields
    if sat_prn[1].startswith("sv_id")
]
prns = [satilite_prn for satilite_prn in prns if satilite_prn != ""]

print(prns)

3.14
30
['30', '20', '13', '14', '11']


In [7]:
print(nmea_sentences_active["GPRMC"])
nmea_parsed = pynmea2.parse(nmea_sentences_active["GPRMC"])
nmea_parsed
GPRMCRecommendedCourse.from_nmea(nmea_parsed)

$GPRMC,200517.00,A,5211.77084,N,00213.26972,W,0.117,,030625,,,A*65


GPRMCRecommendedCourse(timestamp=datetime.datetime(2025, 6, 3, 20, 5, 17, tzinfo=datetime.timezone.utc), status=<GPSStatus.ACTIVE: 'A'>, latitude_nmea='5211.77084', longitude_nmea='00213.26972')

# Process NMEA data stream

In [ ]:
from src.kelder_api.components.gps_new.interface import GPSInterface
from src.kelder_api.configuration.settings import get_settings
from src.kelder_api.components.redis_client.redis_client import RedisClient
from src.kelder_api.components.gps_new.models import GPGSVSatellitesInView, GPSRedisData

redis_client = RedisClient()
gps_interface = GPSInterface(redis_client)

nmea_sentence = pynmea2.parse(nmea_sentences_active["GPGSV_1"])
satellites_in_view = GPGSVSatellitesInView()

satellites_in_view.from_nmea(nmea_sentence)

# i = 1
# satellites_in_view.add_satellite(
#     prn = int(getattr(nmea_sentence, f'sv_prn_num_{i+1}')),
#     elevation = getattr(nmea_sentence, f'elevation_deg_{i+1}'),
#     azimuth = getattr(nmea_sentence, f'azimuth_{i+1}'),
#     snr = float(getattr(nmea_sentence, f'snr_{i+1}')) if getattr(nmea_sentence, f'snr_{i+1}') != "" else None
# )

# await gps_interface.stream_serial_data(list(nmea_sentences_active.values()))
# await gps_interface.stream_serial_data(list(nmea_sentences_void.values()))

Read the latest gps measurement

In [7]:
gps_latest = await gps_interface.read_gps_latest()
# data = await redis_client.read_set("GPS")
gps_latest_active = await gps_interface.read_gps_latest(active=True)
print(f"Latest measurements: {gps_latest.model_dump()}")
print(f"Last active measurement: {gps_latest_active.model_dump()}")

Latest measurements: {'timestamp': datetime.datetime(2025, 6, 9, 20, 22, 34, tzinfo=TzInfo(UTC)), 'status': <GPSStatus.VOID: 'V'>, 'latitude_nmea': '', 'longitude_nmea': 'E', 'active_prn': [], 'hdop': 99.99, 'satellites_in_view': {5: {'prn': 5, 'elevation': 68, 'azimuth': 261, 'snr': None}, 7: {'prn': 7, 'elevation': 37, 'azimuth': 58, 'snr': None}, 8: {'prn': 8, 'elevation': 2, 'azimuth': 52, 'snr': None}, 9: {'prn': 9, 'elevation': 2, 'azimuth': 96, 'snr': None}, 11: {'prn': 11, 'elevation': 4, 'azimuth': 201, 'snr': None}, 13: {'prn': 13, 'elevation': 57, 'azimuth': 276, 'snr': None}, 14: {'prn': 14, 'elevation': 28, 'azimuth': 138, 'snr': 29.0}, 15: {'prn': 15, 'elevation': 21, 'azimuth': 283, 'snr': None}, 18: {'prn': 18, 'elevation': 18, 'azimuth': 322, 'snr': None}, 20: {'prn': 20, 'elevation': 53, 'azimuth': 190, 'snr': 28.0}, 21: {'prn': 21, 'elevation': 56, 'azimuth': 171, 'snr': None}, 22: {'prn': 22, 'elevation': 15, 'azimuth': 152, 'snr': None}, 27: {'prn': 27, 'elevation'

Read GPS history length

In [8]:
gps_latest = await gps_interface.read_gps_history_length(10)
# data = await redis_client.read_set("GPS")
gps_latest_active = await gps_interface.read_gps_history_length(length=10, active=True)
print(f"All measurements: {gps_latest}")
print(f"Active measurements: {gps_latest_active}")

All measurements: [GPSRedisData(timestamp=datetime.datetime(2025, 6, 9, 20, 22, 34, tzinfo=TzInfo(UTC)), status=<GPSStatus.VOID: 'V'>, latitude_nmea='', longitude_nmea='E', active_prn=[], hdop=99.99, satellites_in_view={5: SatelliteInfomation(prn=5, elevation=68, azimuth=261, snr=None), 7: SatelliteInfomation(prn=7, elevation=37, azimuth=58, snr=None), 8: SatelliteInfomation(prn=8, elevation=2, azimuth=52, snr=None), 9: SatelliteInfomation(prn=9, elevation=2, azimuth=96, snr=None), 11: SatelliteInfomation(prn=11, elevation=4, azimuth=201, snr=None), 13: SatelliteInfomation(prn=13, elevation=57, azimuth=276, snr=None), 14: SatelliteInfomation(prn=14, elevation=28, azimuth=138, snr=29.0), 15: SatelliteInfomation(prn=15, elevation=21, azimuth=283, snr=None), 18: SatelliteInfomation(prn=18, elevation=18, azimuth=322, snr=None), 20: SatelliteInfomation(prn=20, elevation=53, azimuth=190, snr=28.0), 21: SatelliteInfomation(prn=21, elevation=56, azimuth=171, snr=None), 22: SatelliteInfomation(

Read timeseries

In [9]:
from datetime import datetime

start_time = datetime(2025, 6, 3, 20, 5, 17)

gps_latest = await gps_interface.read_gps_history_time_series(start_time)
# data = await redis_client.read_set("GPS")
gps_latest_active = await gps_interface.read_gps_history_time_series(
    start_time, active=True
)
print(f"All measurements: {gps_latest}")
print(f"Active measurements {gps_latest_active}")

All measurements: [{'timestamp': '2025-06-09T20:22:34Z', 'status': 'V', 'latitude_nmea': '', 'longitude_nmea': 'E', 'active_prn': [], 'hdop': 99.99, 'satellites_in_view': {'5': {'prn': 5, 'elevation': 68, 'azimuth': 261, 'snr': None}, '7': {'prn': 7, 'elevation': 37, 'azimuth': 58, 'snr': None}, '8': {'prn': 8, 'elevation': 2, 'azimuth': 52, 'snr': None}, '9': {'prn': 9, 'elevation': 2, 'azimuth': 96, 'snr': None}, '11': {'prn': 11, 'elevation': 4, 'azimuth': 201, 'snr': None}, '13': {'prn': 13, 'elevation': 57, 'azimuth': 276, 'snr': None}, '14': {'prn': 14, 'elevation': 28, 'azimuth': 138, 'snr': 29.0}, '15': {'prn': 15, 'elevation': 21, 'azimuth': 283, 'snr': None}, '18': {'prn': 18, 'elevation': 18, 'azimuth': 322, 'snr': None}, '20': {'prn': 20, 'elevation': 53, 'azimuth': 190, 'snr': 28.0}, '21': {'prn': 21, 'elevation': 56, 'azimuth': 171, 'snr': None}, '22': {'prn': 22, 'elevation': 15, 'azimuth': 152, 'snr': None}, '27': {'prn': 27, 'elevation': 4, 'azimuth': 19, 'snr': None},

Read all history:

In [10]:
gps_latest = await gps_interface.read_gps_all_history()
# data = await redis_client.read_set("GPS")
gps_latest_active = await gps_interface.read_gps_all_history(active=True)
print(f"All measurements: {gps_latest}")
print(f"Active measurements {gps_latest_active}")

All measurements: [{'timestamp': '2025-06-09T20:22:34Z', 'status': 'V', 'latitude_nmea': '', 'longitude_nmea': 'E', 'active_prn': [], 'hdop': 99.99, 'satellites_in_view': {'5': {'prn': 5, 'elevation': 68, 'azimuth': 261, 'snr': None}, '7': {'prn': 7, 'elevation': 37, 'azimuth': 58, 'snr': None}, '8': {'prn': 8, 'elevation': 2, 'azimuth': 52, 'snr': None}, '9': {'prn': 9, 'elevation': 2, 'azimuth': 96, 'snr': None}, '11': {'prn': 11, 'elevation': 4, 'azimuth': 201, 'snr': None}, '13': {'prn': 13, 'elevation': 57, 'azimuth': 276, 'snr': None}, '14': {'prn': 14, 'elevation': 28, 'azimuth': 138, 'snr': 29.0}, '15': {'prn': 15, 'elevation': 21, 'azimuth': 283, 'snr': None}, '18': {'prn': 18, 'elevation': 18, 'azimuth': 322, 'snr': None}, '20': {'prn': 20, 'elevation': 53, 'azimuth': 190, 'snr': 28.0}, '21': {'prn': 21, 'elevation': 56, 'azimuth': 171, 'snr': None}, '22': {'prn': 22, 'elevation': 15, 'azimuth': 152, 'snr': None}, '27': {'prn': 27, 'elevation': 4, 'azimuth': 19, 'snr': None},